# Municode Zoning-Code Collector

Retrieve publicly accessible municipal-code content from
Municode's web application endpoints for research and source inventory.

It does **not** determine what zoning rules mean or create a score.

Workflow:

1. Find the municipality's Municode client identifier.
2. Inspect available products and identify the Code of Ordinances product.
3. Traverse the table of contents.
4. Identify zoning, land-use, planning, and subdivision chapters.
5. Retrieve source content for selected nodes.
6. Save raw JSON, HTML/text, source metadata, and retrieval logs locally.
7. Manually verify that collected content is current and official before analysis.

---

## Municode API Test — Greenwich Zoning Quote

Simple proof-of-concept: hit Municode's own (public, unauthenticated) full-text search
API directly, for Greenwich, CT, and pull back a literal quoted passage that mentions
lot size/area. This is a connectivity test, not a legal determination — the quote still
needs human review before anyone relies on it.


In [1]:
import re
import requests

HEADERS = {"User-Agent": "district-zoning-monitor/0.1 (research collection)"}

# Terms we're hoping to find in the code, in priority order.
LOT_SIZE_TERMS = ["minimum lot size", "minimum lot area", "minimum building lot"]


def municode_client_id(state_abbr: str, municipality: str) -> int:
    """Look up a municipality's Municode client ID from its organization record.

    Example: municode_client_id("ct", "greenwich") -> 8693
    """
    url = f"https://library.municode.com/localapi/Organizations/GetByUrlEncodedNames/{state_abbr}/{municipality}"
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.json()["ClientID"]


def municode_search(client_id: int, state_id: int, search_text: str, page_size: int = 10):
    """Call Municode's public full-text search API and return the raw hit list.

    This is the same endpoint library.municode.com's own search box uses, so it
    requires no login/API key and returns real text fragments straight from the
    code, each with its own section id and chapter/article breadcrumb.
    """
    url = "https://library.municode.com/localapi/search"
    params = {
        "clientId": client_id,
        "contentTypeId": "CODES",
        "fragmentSize": 500,   # how much surrounding text to return per hit
        "isAdvanced": "false",
        "isAutocomplete": "false",
        "mode": "CLIENTMODE",
        "pageNum": 1,
        "pageSize": page_size,
        "searchText": search_text,
        "sort": 0,
        "stateId": state_id,
        "titlesOnly": "false",
    }
    resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json().get("hits", [])


def pick_best_hit(hits):
    """Score each hit by how zoning-relevant its chapter/article path and matched
    text look, and return the highest scorer (ties go to whichever came first).

    "Land Use" chapter and "Building Zone Regulations" mentions outrank a generic
    "zon"/"planning" hint, since those alone also match unrelated sections like
    the Planning & Zoning Board of Appeals' general filing procedures.
    """
    def score(hit):
        breadcrumb = " ".join(a["title"] for a in hit["ancestors"]).lower()
        fragment = hit["contentFragment"].lower()
        s = 0
        if "land use" in breadcrumb:
            s += 3
        if "building zone regulations" in fragment:
            s += 3
        if any(h in breadcrumb for h in ("zon", "planning", "building")):
            s += 1
        return s

    if not hits:
        return None
    return max(hits, key=score)


def clean_quote(html_fragment: str) -> str:
    """Strip Municode's <em class="hit..."> highlight tags and collapse whitespace,
    so the quote reads as plain literal text."""
    text = re.sub(r"<[^>]+>", "", html_fragment)
    return re.sub(r"\s+", " ", text).strip()


In [2]:
# Look up Greenwich's Municode client ID, then try each search term until we get a hit.
client_id = municode_client_id("ct", "greenwich")
state_id = 7  # Connecticut, per Municode's own org record above

best_hit, winning_term = None, None
for term in LOT_SIZE_TERMS:
    hits = municode_search(client_id, state_id, term)
    if hits:
        best_hit = pick_best_hit(hits)
        winning_term = term
        break

if best_hit is None:
    print("No candidate section found for Greenwich via the Municode search API.")
else:
    breadcrumb = " > ".join(a["title"] for a in best_hit["ancestors"])
    view_url = f"https://library.municode.com/ct/greenwich/codes/code_of_ordinances?nodeId={best_hit['nodeId']}"

    print("MUNICODE API TEST — GREENWICH, CT")
    print(f"Search term used: \"{winning_term}\"")
    print(f"Section: {best_hit['title']}")
    print(f"Chapter/article path: {breadcrumb}")
    print(f"Municode node id: {best_hit['nodeId']}")
    print(f"View on Municode: {view_url}")
    print("Literal quote (needs human review — candidate evidence only):")
    print(f'  "{clean_quote(best_hit["contentFragment"])}"')


MUNICODE API TEST — GREENWICH, CT
Search term used: "minimum lot size"
Section: Sec. 6-234. - General provisions.
Chapter/article path: Greenwich, Connecticut - Code of Ordinances > CHAPTER 6. LAND USE > ARTICLE 3. - INLAND WETLANDS AND WATERCOURSES.
Municode node id: CH6.LAUS_ART3INWEWA_S6-234GEPR
View on Municode: https://library.municode.com/ct/greenwich/codes/code_of_ordinances?nodeId=CH6.LAUS_ART3INWEWA_S6-234GEPR
Literal quote (needs human review — candidate evidence only):
  "Each applicant shall pay a minimum filing fee of fifty dollars ($50.) for each application at the time said application is filed. For residential property and uses, an additional fee of forty dollars ($40.) shall be required for each lot of a subdivision or, where no subdivision is involved, for each acre or any part of an acre of the subject parcel in excess of the minimum size of building lot required in the building zone regulations"
